# [v2] XTTS-v2 복제·채점 — XTTS 대리 모델 (scale 9)

`notebooks/03_clone_xtts.ipynb`에서 5단계 경로를 바꾸고 C1·C4 재사용 칸을 넣은 버전이다. 결과: `runs/enkidu_xtts_c9/`.

# XTTS-v2 복제 + 화자 인증 — Enkidu 평가셋 (화자 4명 / 63개)

같은 63개 클립에서 만든 네 입력 세트를 각각 XTTS-v2의 참조 음성으로 줘서 복제하고, **D1 원본을 기준**으로 복제본이 얼마나 같은 사람으로 인증되는지 ECAPA와 Resemblyzer로 비교한다.

| 입력 (각 63개) | 의미 | 복제본 |
| --- | --- | --- |
| **D1** | LibriSpeech 원본 | **C1** 공격이 원래 얼마나 잘 되는가 |
| **D2** | Enkidu 보호본 | **C2** 보호가 복제를 얼마나 막는가 |
| **D3** | D2를 De-AntiFake로 정화 | **C3** 정화로 공격이 얼마나 회복되는가 |
| **D4** | D1을 De-AntiFake로 정화 | **C4** 정화 모델 자체의 영향 (상한) |

```
D1 ─► XTTS-v2 ─► C1 ┐
D2 ─► XTTS-v2 ─► C2 ├─► D1 원본과 비교 (ECAPA, Resemblyzer) ─► 쌍체 분석
D3 ─► XTTS-v2 ─► C3 ┤
D4 ─► XTTS-v2 ─► C4 ┘
```

## 원본 가이드에서 바꾼 것
- **화자별 하위 폴더 지원.** D1~D4가 `D<n>/<화자>/<파일>.wav` 구조다. 파일명(`libri_p1089-0042`)이 전역에서 유일하므로 항목 id로 그대로 쓰고, 작업 폴더에서는 평탄하게 둔다.
- **문장 3개 → 2개.** 63항목 × 4조건 × 3문장 = 756개는 T4에서 한 시간 반이 넘는다. 2문장이면 504개, 약 50분. `cfg["sentences"]`에서 조절한다.
- **EER을 실제로 계산한다.** 화자당 클립이 13~18개라 같은 화자 쌍이 475개 생긴다. 원본 가이드는 화자당 1개를 가정해 EER을 포기했지만 우리는 낼 수 있다.
- **화자 4명 경고 추가.** 항목은 63개지만 독립 단위는 화자 4명이다. 화자 단위 부트스트랩 신뢰구간은 믿기 어려우므로 항목 단위 구간을 함께 낸다.

## 데이터 구조 (`내 드라이브/xtts_exp/`)
```
D1/<화자>/libri_p<화자>-<번호>.wav   원본 (206개 중 63개만 쓰임)
D2/<화자>/...                        Enkidu 보호본 63개
D3/<화자>/...                        D2 정화본 63개
D4/<화자>/...                        D1 정화본 63개 (대조군)
```
네 세트에 **모두 있는 파일만** 항목으로 쓴다. D1에 206개가 있어도 D2에 있는 63개만 교집합으로 남는다.

## 실행 전
런타임 → 런타임 유형 변경 → **T4 GPU**

처음에는 6단계에서 `N_ITEMS = 3`으로 시험해 한 개당 생성 시간을 재고, 그다음 `None`으로 전체를 돌린다. 복제 셀은 **이미 만들어진 정상 파일을 건너뛰므로** 중간에 끊겨도 이어서 돌릴 수 있다.

## 1단계. 코랩에 conda 설치
실행 후 런타임이 **자동으로 재시작**됩니다. 정상이에요. 재시작 후에는 이 칸을 다시 실행하지 말고 2단계부터 하세요.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2단계. 환경 두 개 만들기 (XTTS용 `xtts`, 화자 인증용 `sv`). 몇 분 걸려요
- 파이썬 3.10 환경을 conda로 따로 만듭니다(코랩 기본 3.13에서는 `TTS` 패키지가 지원되지 않을 가능성이 커서, 이 지원 범위는 기억 기반이라 미확인).
- `torch==2.5.1`, `torchaudio==2.5.1`: 최신 torch(2.6 이상)에서는 XTTS 체크포인트 로딩이 막힌다는 보고를 기억해서 고정(미확인).
- `transformers<4.50`: 최신 `transformers`와 `TTS 0.22`의 조합 오류를 피하려는 고정(미확인).
- `setuptools<81`: `pkg_resources` 오류를 막습니다(이전에 실제로 겪음).

In [ ]:
!conda --version
!conda create -y -n xtts python=3.10
!conda run -n xtts --no-capture-output pip install "TTS==0.22.0" "torch==2.5.1" "torchaudio==2.5.1" "transformers<4.50" "setuptools<81"

In [ ]:
!conda create -y -n sv python=3.10
!conda run -n sv --no-capture-output pip install -q torch torchaudio speechbrain librosa scipy soundfile pandas "setuptools<81" webrtcvad-wheels
!conda run -n sv --no-capture-output pip install -q resemblyzer --no-deps

## 3단계. 설치 확인

In [ ]:
!conda run -n xtts --no-capture-output python -c "import TTS, torch; print('TTS', TTS.__version__, '| torch', torch.__version__); print('GPU:', torch.cuda.is_available())"
!conda run -n sv --no-capture-output python -c "from resemblyzer import VoiceEncoder; import speechbrain; print('sv ok')"

`TTS 0.22.0 | torch 2.5.1`, `GPU: True`, `sv ok`가 나오면 성공입니다.

## 4단계. XTTS-v2 약관(CPML) 동의 확인
XTTS-v2는 **비상업용 CPML** 라이선스입니다. 처음 내려받을 때 약관 동의를 묻는 것으로 기억하는데(미확인), 코랩에서는 입력을 줄 수 없어서 환경변수로 동의를 전달해야 합니다.
**약관(https://coqui.ai/cpml)을 직접 읽고 동의할 때만** 아래 `AGREE_CPML`을 `True`로 바꾸세요. 동의하지 않으면 이 칸에서 멈춥니다.

In [ ]:
AGREE_CPML = True   # <- 약관을 읽고 동의하면 True로 바꾸세요

assert AGREE_CPML, "CPML 약관(https://coqui.ai/cpml)을 읽고 동의하는 경우에만 AGREE_CPML = True 로 바꾸고 다시 실행하세요."
import os
os.environ["COQUI_TOS_AGREED"] = "1"                  # 약관 동의 전달
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # torch 신버전에서의 체크포인트 로딩 오류 방지(미확인)
print("설정 완료")

## 5단계. 데이터 위치 정하기 (구글 드라이브 권장)
25개 x 3세트 = 75개 파일이라 **구글 드라이브**를 권장합니다. 세션이 초기화돼도 복제 결과가 남아서 **이어서 실행**할 수 있습니다.

드라이브에 이런 구조로 올려 두세요.
```
내 드라이브/xtts_exp/
 ├ D1/   원본 25개             예: p1089-0001.wav ...
 ├ D2/   보호 섭동 적용 25개      파일 이름이 D1과 같아야 함
 ├ D3/   정화(De-AntiFake) 25개  파일 이름이 D1과 같아야 함
 └ D4/   D1을 De-AntiFake로 정화 25개 (선택) 파일 이름이 D1과 같아야 함
```
- **D4는 D3를 만들 때와 똑같은 De-AntiFake 설정**(체크포인트, 정화 스텝 `t`, 정제 단계 사용 여부, 전사 입력 방식)으로 만들어야 비교가 됩니다.
- D2, D3, D4는 **wav 또는 flac**으로 두세요. mp3 등으로 저장하면 압축이 섭동을 지울 수 있습니다.
- SceneGuard와 De-AntiFake의 출력은 입력과 같은 파일 이름을 쓰므로 보통 이름이 맞습니다. 꼬리표가 붙었다면 아래 `STRIP`으로 제거합니다.
- **보호 기법이 여러 개면** D2, D3 폴더만 바꾸고 `RUN_NAME`을 바꿔서 (D4는 보호 기법과 무관하므로 공통) 이 노트북을 기법마다 한 번씩 실행하세요(D1은 공통).
- 영어, 한 사람 목소리, 5~12초 정도의 클립이 안전합니다(모델 카드는 약 6초를 언급).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/xtts_exp"
RUN_NAME   = "enkidu_xtts_c9"          # 보호 기법별로 바꾼다 (safespeech_63, sceneguard_63 ...)
FOLDER = {"D1": "D1",
          "D2": "surrogate_exp/xtts/D2_c9",
          "D3": "surrogate_exp/xtts/D3_c9",
          "D4": "D4_new"}
USE_D4     = True                 # D4(=De-AntiFake(D1)) 폴더가 없으면 False
OUT_DIR    = f"{DRIVE_ROOT}/runs/{RUN_NAME}/clones"
os.makedirs(OUT_DIR, exist_ok=True)

CONDS_IN = ["D1", "D2", "D3"] + (["D4"] if USE_D4 else [])
CLONE_OF = {d: "C" + d[1:] for d in CONDS_IN}
CONDS_CL = [CLONE_OF[d] for d in CONDS_IN]

for c in CONDS_IN:
    p = Path(f"{DRIVE_ROOT}/{FOLDER.get(c, c)}")
    n_spk = len([d for d in p.iterdir() if d.is_dir()]) if p.is_dir() else 0
    n_wav = len(list(p.glob("*/*.wav"))) if p.is_dir() else 0
    print(f"{c} -> {p} | 존재: {p.is_dir()} | 화자 {n_spk}명 | wav {n_wav}개")
    assert p.is_dir(), f"{c} 폴더가 없습니다." + (" D4가 없으면 USE_D4 = False" if c == "D4" else "")

print("\n출력:", OUT_DIR)

**C1·C4 재사용.** C1(D1 복제)과 C4(D4 복제)는 보호 기법과 무관하므로 v0 실행 결과를 복사한다. 복제는 C2·C3 252개만 새로 만든다. `{'C1': 126, 'C2': 0, 'C3': 0, 'C4': 126}`이 나와야 한다.

In [ ]:
import shutil, glob, os
SRC = f"{DRIVE_ROOT}/runs/enkidu_63_d4new/clones"     # v0 결과 (읽기만)
n = 0
for c in ("C1", "C4"):
    for p in glob.glob(f"{SRC}/{c}__*.wav"):
        dst = os.path.join(OUT_DIR, os.path.basename(p))
        if not os.path.exists(dst):
            shutil.copy2(p, dst); n += 1
print("복사:", n, "개")
print({c: len(glob.glob(f"{OUT_DIR}/{c}__*.wav")) for c in ("C1","C2","C3","C4")})

(대안) 드라이브 대신 zip을 올릴 때는 아래 칸의 `#`을 지우고 쓰세요. 이 경우 세션이 끊기면 결과가 사라집니다.

In [ ]:
# import subprocess
# from google.colab import files
# up = files.upload()    # D1.zip, D2.zip, D3.zip (+ D4.zip)
# for c in CONDS_IN:
#     subprocess.run(["unzip", "-q", "-o", f"{c}.zip", "-d", f"/content/upload/{c}"], check=True)
# DRIVE_ROOT = "/content/upload"

## 6단계. 입력 정리: 교집합 추리기, 16kHz 변환, 검사

- `D<n>/<화자>/<파일>.wav`를 순회한다. 파일명이 전역에서 유일하므로 **항목 id는 파일명**(`libri_p1089-0042`)을 쓴다.
- 네 세트에 **모두 있는 파일만** 항목이 된다. D1에 206개가 있어도 D2 기준 63개로 줄어든다.
- 화자는 파일명의 `p숫자-` 패턴에서 뽑는다 (`libri_p1089-0042` → 화자 `1089`).
- **파일을 두 벌 만든다.** `work/ref_D*`는 복제 참조용(원래 샘플레이트 유지, 모노, 32bit float — 섭동 보존), `work/D*`는 채점용(16kHz 모노).

처음에는 `N_ITEMS = 3`으로 시험하고, 전체를 돌릴 때 `None`으로 바꾼다.

In [ ]:
import os, re, subprocess
from pathlib import Path
import soundfile as sf

N_ITEMS = None             # 시험: 3 / 전체: None  (본 실험은 반드시 None)
MAX_SEC = None
STRIP = {c: "" for c in CONDS_IN}
SPEAKER_OVERRIDE = {}
EXTS  = (".wav", ".mp3", ".m4a", ".mp4", ".flac", ".ogg")
LOSSY = (".mp3", ".m4a", ".mp4", ".ogg")
SRC_DIR = {c: f"{DRIVE_ROOT}/{FOLDER.get(c, c)}" for c in CONDS_IN}
W = "/content/work"

def stem_of(fn, cond):
    s = os.path.splitext(fn)[0]
    tail = STRIP[cond]
    return s[:-len(tail)] if tail and s.endswith(tail) else s

# 화자별 하위 폴더를 순회해 {파일명: 전체경로}로 모은다
files = {}
for c in CONDS_IN:
    d = {}
    for spk_dir in sorted(Path(SRC_DIR[c]).iterdir()):
        if not spk_dir.is_dir():
            continue
        for f in sorted(spk_dir.iterdir()):
            if f.suffix.lower() in EXTS:
                d[stem_of(f.name, c)] = str(f)
    files[c] = d
    print(f"{c} 파일 수: {len(d)}")

allstems = set().union(*[set(v) for v in files.values()])
ITEMS = sorted(set.intersection(*[set(files[c]) for c in CONDS_IN]))
print(f"\n교집합: {len(ITEMS)}개 (전체 이름 종류 {len(allstems)}개)")
for c in CONDS_IN:
    miss = sorted(allstems - set(files[c]))
    if miss:
        print(f"  {c}에 없는 항목 {len(miss)}개:", miss[:5], "..." if len(miss) > 5 else "")

assert ITEMS, "네 세트에 공통으로 있는 파일이 없습니다. 파일명이 같은지 확인하세요."
for s in ITEMS:
    assert "__" not in s and "." not in s, f"파일 이름에 '__' 또는 '.'가 있으면 안 됩니다: {s}"

if N_ITEMS:
    ITEMS = ITEMS[:N_ITEMS]
    print(f"\n[시험 모드] {len(ITEMS)}개만 사용합니다. 전체는 N_ITEMS = None")
print("사용 항목 수:", len(ITEMS))

for c in CONDS_IN:
    n = sum(files[c][s].lower().endswith(LOSSY) for s in ITEMS)
    if n and c != "D1":
        print(f"경고: {c}에 손실 압축 파일 {n}개. 압축이 섭동을 지웠을 수 있습니다.")

durs, srs = {}, {}
for c in CONDS_IN:
    os.makedirs(f"{W}/{c}", exist_ok=True)
    os.makedirs(f"{W}/ref_{c}", exist_ok=True)
    for s in ITEMS:
        src = files[c][s]

        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vn", "-ar", "16000", "-ac", "1"]
        if MAX_SEC is not None:
            cmd += ["-t", str(MAX_SEC)]
        cmd += [f"{W}/{c}/{s}.wav"]            # (1) 채점용 16kHz
        subprocess.run(cmd, check=True)

        ref = f"{W}/ref_{c}/{s}.wav"
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vn", "-ac", "1"]
        if MAX_SEC is not None:
            cmd += ["-t", str(MAX_SEC)]
        cmd += ["-c:a", "pcm_f32le", ref]      # (2) 복제 참조용 원본 샘플레이트
        subprocess.run(cmd, check=True)

        info = sf.info(ref)
        durs[(c, s)] = info.duration
        srs[(c, s)] = info.samplerate

print("\n복제 참조 샘플레이트:", {c: sorted({srs[(c, s)] for s in ITEMS}) for c in CONDS_IN})

short = sorted({s for (c, s), d in durs.items() if d < 6})
if short:
    print(f"주의: 6초 미만 클립 {len(short)}개 - XTTS 참조로는 짧습니다:", short[:5])

diff = [s for s in ITEMS
        if max(durs[(c, s)] for c in CONDS_IN) - min(durs[(c, s)] for c in CONDS_IN) > 0.3]
if diff:
    print(f"주의: 세트 간 길이가 0.3초 넘게 다른 항목 {len(diff)}개:", diff[:5])

def speaker_of(stem):
    if stem in SPEAKER_OVERRIDE:
        return SPEAKER_OVERRIDE[stem]
    m = re.search(r"p(\d+)-", stem)
    return m.group(1) if m else stem

SPK = {s: speaker_of(s) for s in ITEMS}
import collections
print("\n화자별 항목 수:", dict(collections.Counter(SPK.values())))
print("화자 수:", len(set(SPK.values())), "/ 항목 수:", len(ITEMS))

## 7단계. 입력 음성 들어 보기 (첫 항목)

In [ ]:
from IPython.display import Audio, display
s0 = ITEMS[0]
print("항목:", s0)
for c in CONDS_IN:
    print(c); display(Audio(filename=f"{W}/ref_{c}/{s0}.wav"))

## 8단계. 복제 설정

- 문장을 **모든 조건에서 똑같이** 쓴다. 조건 간 차이만 남기기 위해서다.
- 시드는 (항목, 문장, 반복)으로 고정하고 **조건과 무관하게** 같게 준다. GPU 연산 특성상 완전한 결정성은 보장되지 않는다.
- 참조는 `work/ref_D*`(원래 샘플레이트). 16kHz로 내리면 8kHz 위 섭동이 잘려 보호 효과가 과소평가된다.
- 이미 만들어진 **정상 파일만** 건너뛴다. 끊겨도 다시 돌리면 이어서 생성한다.
- `manifest.json`에 참조 음성 해시와 설정을 기록해, 같은 `RUN_NAME`에서 입력이나 문장이 바뀌면 멈춘다. 그때는 `RUN_NAME`을 바꾼다.

**생성 개수와 시간**

| 문장 수 | 생성 개수 (63항목 × 4조건) | T4 예상 |
| --- | --- | --- |
| 1 | 252 | 약 25분 |
| 2 (기본) | 504 | 약 50분 |
| 3 | 756 | 약 1시간 20분 |

문장을 늘려도 표본의 독립 단위는 늘지 않는다(같은 항목의 반복 측정이라 14단계에서 먼저 평균한다). 문장 하나짜리 우연을 막는 보험이므로 2개면 충분하다. 시간이 빠듯하면 1개로 줄여도 결론은 바뀌지 않는다.

In [ ]:
import json
REPEATS = 1          # 시간이 되면 2~3 (생성 변동 확인용)

cfg = {
    "out_dir": OUT_DIR,
    "language": "en",
    "repeats": REPEATS,
    "sentences": [
        "The weather is nice today, so I decided to take a walk.",
        "My favorite season is autumn because the leaves turn red and yellow.",
    ],
    "items": ITEMS,
    "conds": CONDS_CL,
    "refs": {CLONE_OF[d]: f"{W}/ref_{d}" for d in CONDS_IN},
}
json.dump(cfg, open("/content/clone_cfg.json", "w"), ensure_ascii=False, indent=1)

total = len(ITEMS) * len(CONDS_CL) * len(cfg["sentences"]) * cfg["repeats"]
print(f"만들 복제 개수: {total}")
print(f"  항목 {len(ITEMS)} × 조건 {len(CONDS_CL)} × 문장 {len(cfg['sentences'])} × 반복 {cfg['repeats']}")
print(f"  T4에서 개당 6초 가정 시 약 {total * 6 / 60:.0f}분 (+ 첫 실행은 모델 내려받기)")

In [ ]:
%%writefile /content/clone_batch.py
import os, sys, json, time, zlib, random, hashlib
import numpy as np, torch, soundfile as sf
from TTS.api import TTS

MODEL = "tts_models/multilingual/multi-dataset/xtts_v2"
cfg = json.load(open(sys.argv[1]))
od = cfg["out_dir"]; os.makedirs(od, exist_ok=True)

def audio_hash(p):                       # 파일 헤더가 아닌 샘플 값으로 해시 (ffmpeg 버전 차이에 무관)
    y, sr = sf.read(p, dtype="float32")
    return hashlib.md5(y.tobytes() + str(sr).encode()).hexdigest()

def valid(p):
    try:
        i = sf.info(p); return i.frames > 0 and i.duration > 0.3
    except Exception:
        return False

# ---- 실행 조건 기록/검사: 같은 RUN_NAME에서 입력이나 설정이 바뀌면 중단 ----
settings = {"model": MODEL, "language": cfg["language"], "sentences": cfg["sentences"],
            "seed": "crc32(item|sentence|repeat)"}
mp = os.path.join(od, "manifest.json")
man = json.load(open(mp)) if os.path.exists(mp) else {"settings": None, "refs": {}}
if man["settings"] is not None and man["settings"] != settings:
    sys.exit("중단: 이 RUN_NAME의 기존 복제본과 설정(모델/언어/문장)이 다릅니다. RUN_NAME을 바꾸세요.")
conflict = []
for cond in cfg["conds"]:
    for item in cfg["items"]:
        key, h = f"{cond}/{item}", audio_hash(os.path.join(cfg["refs"][cond], item + ".wav"))
        if man["refs"].get(key, h) != h:
            conflict.append(key)
        man["refs"][key] = h
if conflict:
    sys.exit(f"중단: 참조 음성이 이전 실행과 다른 항목 {len(conflict)}개 {conflict[:5]}. RUN_NAME을 바꾸세요.")
man["settings"] = settings
json.dump(man, open(mp, "w"), ensure_ascii=False, indent=1)

tts = TTS(MODEL, gpu=True)

def seed_for(item, i, r):
    s = zlib.crc32(f"{item}|{i}|{r}".encode()) % (2**31)
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

failed, done, t0 = [], 0, time.time()
for item in cfg["items"]:
    for cond in cfg["conds"]:
        ref = os.path.join(cfg["refs"][cond], item + ".wav")
        for i, text in enumerate(cfg["sentences"], 1):
            for r in range(1, cfg["repeats"] + 1):
                out = os.path.join(od, f"{cond}__{item}__s{i}__r{r}.wav")
                if os.path.exists(out) and valid(out):
                    continue
                tmp = out + ".part"              # 끝까지 생성된 파일만 .wav로 이름을 바꿈
                try:
                    seed_for(item, i, r)
                    tts.tts_to_file(text=text, file_path=tmp, speaker_wav=ref, language=cfg["language"])
                    if not valid(tmp):
                        raise RuntimeError("생성 파일이 비었거나 읽을 수 없음")
                    os.replace(tmp, out)
                    done += 1
                    if done <= 3 or done % 10 == 0:
                        print(f"[{done}] {os.path.basename(out)}  경과 {time.time()-t0:.0f}초", flush=True)
                except Exception as e:
                    if os.path.exists(tmp): os.remove(tmp)
                    failed.append({"out": out, "error": repr(e)[:300]})
                    print("실패:", os.path.basename(out), repr(e)[:200], flush=True)
json.dump(failed, open(os.path.join(od, "failed.json"), "w"), ensure_ascii=False, indent=1)
print(f"완료: 새로 만든 {done}개, 실패 {len(failed)}개, 총 {time.time()-t0:.0f}초")

**복제 실행.** 처음에는 XTTS-v2 모델을 내려받느라(큰 파일, 정확한 크기는 미확인) 시간이 걸립니다. 첫 3개가 만들어지는 데 걸린 시간을 보고 전체 소요 시간을 가늠하세요. (실측: 모델 내려받기 포함 첫 실행 약 17분)

In [ ]:
!conda run -n xtts --no-capture-output python /content/clone_batch.py /content/clone_cfg.json

## 9단계. 복제 결과 확인

In [ ]:
import glob, os, collections, json
import soundfile as sf
from IPython.display import Audio, display
expected = {f"{c}__{s}__s{i}__r{r}.wav" for c in CONDS_CL for s in ITEMS
            for i in range(1, len(cfg["sentences"]) + 1) for r in range(1, cfg["repeats"] + 1)}
present = {os.path.basename(f) for f in glob.glob(f"{OUT_DIR}/*.wav")}
bad = []
for f in sorted(expected & present):
    try:
        i = sf.info(f"{OUT_DIR}/{f}")
        if i.frames == 0 or i.samplerate != 24000:
            bad.append((f, i.samplerate, i.frames))
    except Exception as e:
        bad.append((f, "읽기 실패", repr(e)[:80]))
missing = sorted(expected - present)
cnt = collections.Counter(f.split("__")[0] for f in expected & present)
print(f"조건별 복제 개수 (조건당 기대값 {len(expected)//len(CONDS_CL)}):", dict(cnt))
print("누락:", len(missing), missing[:5]); print("이상 파일(빈 파일/24kHz 아님/읽기 실패):", len(bad), bad[:5])
extra = sorted(present - expected)
if extra:
    print(f"참고: 이번 설정 밖의 파일 {len(extra)}개가 폴더에 있습니다(채점에서 제외됨):", extra[:3])
fp = f"{OUT_DIR}/failed.json"
if os.path.exists(fp):
    fl = json.load(open(fp)); print("실패 기록:", len(fl), "개", fl[:2])
s0 = ITEMS[0]
for c in CONDS_CL:
    p = f"{OUT_DIR}/{c}__{s0}__s1__r1.wav"
    if os.path.exists(p):
        print(c, "(문장 1)"); display(Audio(filename=p))

## 10단계. 채점 스크립트 저장
- **ECAPA**: SpeechBrain `spkrec-ecapa-voxceleb` 임베딩의 코사인 점수
- **Resemblyzer**: d-vector 코사인 점수
- **주 결과는 연속 점수와 조건 간 쌍체 차이**입니다. 통과/탈락은 보조 지표입니다.
- `*_pass_ref` 열의 0.25 / 0.697은 **참고선일 뿐 보정된 임계값이 아닙니다.**
  - 0.25는 SpeechBrain `verify_batch`의 기본값으로, 특정 모델·데이터에서 EER로 정한 값이 아닙니다. 다른 연구(VocalBridge)는 같은 계열 모델의 EER 임계값을 ECAPA 0.419, x-vector 0.951로 보고해, 모델마다 크게 다릅니다.
  - 0.697은 De-AntiFake가 자기 평가셋에서 d-vector EER로 정한 값이라 Resemblyzer와 계열은 같지만 우리 데이터에서 검증되지 않았습니다.
  - 우리 데이터 기준 임계값은 12단계에서 **다른 화자 분포의 FAR 5%**로 따로 계산합니다.
- 두 모델의 점수는 서로 다른 척도라 모델끼리 비교하지 마세요.
- 실행 첫머리에 `verify_files`와 직접 계산 점수를 대조하는 검증 줄을 출력합니다.

In [ ]:
%%writefile /content/sv_eval2.py
import sys, csv, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, soundfile as sf, torch, torchaudio

cfg = json.load(open(sys.argv[1]))

from speechbrain.inference.speaker import SpeakerRecognition
ver = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                      savedir="/content/models/ecapa_sv")
from resemblyzer import VoiceEncoder, preprocess_wav
enc = VoiceEncoder()

def load16(path):
    y, sr = sf.read(path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    w = torch.from_numpy(y).unsqueeze(0)
    if sr != 16000:
        w = torchaudio.functional.resample(w, sr, 16000)
    return w

_cache = {}
def emb(path):
    if path not in _cache:
        with torch.no_grad():
            e = ver.encode_batch(load16(path), normalize=False).squeeze().cpu().numpy()
        r = enc.embed_utterance(preprocess_wav(path))
        _cache[path] = (e, r)
    return _cache[path]

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = cfg["pairs"]
first = pairs[0]
sc, _ = ver.verify_files(first["ref"], first["test"])
mine = cos(emb(first["ref"])[0], emb(first["test"])[0])
print(f"[검증] verify_files={sc.item():.4f} / 직접 계산={mine:.4f} (거의 같아야 정상)", flush=True)

rows = []
for n, p in enumerate(pairs, 1):
    if n % 500 == 0:
        print(f"  {n}/{len(pairs)}", flush=True)
    e1, r1 = emb(p["ref"]); e2, r2 = emb(p["test"])
    s_e = cos(e1, e2); s_r = cos(r1, r2)
    rows.append({**{k: p.get(k, "") for k in ("kind", "cond", "item", "ref_item", "speaker", "sentence", "repeat")},
                 "ecapa": round(s_e, 4), "ecapa_pass_ref": s_e >= 0.25,
                 "resemblyzer": round(s_r, 4), "resemblyzer_pass_ref": s_r >= 0.697})

cols = ["kind", "cond", "item", "ref_item", "speaker", "sentence", "repeat",
        "ecapa", "ecapa_pass_ref", "resemblyzer", "resemblyzer_pass_ref"]
with open(cfg["out_csv"], "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(rows)
print("saved", cfg["out_csv"], "rows:", len(rows))

## 11단계. 비교 쌍 만들고 채점 실행
| kind | 의미 | 용도 |
|---|---|---|
| `clone` | C1~C4 복제본 vs **D1의 같은 항목 원본** | 본 비교 |
| `input` | D1~D4 입력 음성 자체 vs D1 원본 | 보호·정화가 입력 신호에 준 변화 (복제 전) |
| `clone_diff_speaker` | **다른 화자의 C1 복제본** vs 대상 화자 D1 원본 | 같은 합성 도메인의 "다른 사람이면 이 정도" 기준선, FAR 임계값 계산용 (화자 25명이면 25x24x문장 수) |
| `real_diff_speaker` | D1 원본 vs **다른 화자**의 D1 원본 (전체 조합) | 실제 녹음 기준 "다른 사람" 기준선 |
| `real_same_speaker` | D1 원본 vs **같은 화자의 다른 클립** | 같은 화자 클립이 여러 개일 때만 생김. 화자당 1개면 0개(EER 계산 불가) |

임베딩은 파일마다 한 번만 계산하므로 `clone_diff_speaker`가 많아도 시간은 거의 늘지 않습니다.

In [ ]:
import collections
import os, glob, json, random, itertools
pairs = []
clone_files = collections.defaultdict(list)          # (cond, item) -> 복제 파일들
for p in sorted(glob.glob(f"{OUT_DIR}/*.wav")):
    parts = os.path.basename(p)[:-4].split("__")
    if len(parts) != 4 or parts[0] not in CONDS_CL or parts[1] not in SPK:
        continue
    cond, item, s, r = parts
    if int(s[1:]) > len(cfg["sentences"]) or int(r[1:]) > cfg["repeats"]:
        continue
    clone_files[(cond, item)].append((p, s[1:], r[1:]))
    pairs.append(dict(kind="clone", cond=cond, item=item, speaker=SPK[item], sentence=s[1:], repeat=r[1:],
                      test=p, ref=f"{W}/D1/{item}.wav"))
for cond in CONDS_IN:
    for item in ITEMS:
        pairs.append(dict(kind="input", cond=cond, item=item, speaker=SPK[item],
                          test=f"{W}/{cond}/{item}.wav", ref=f"{W}/D1/{item}.wav"))
# 다른 화자의 C1 복제본 vs 대상 화자 원본
for a in ITEMS:
    for b in ITEMS:
        if SPK[a] == SPK[b]:
            continue
        for p, s, r in clone_files[("C1", b)]:
            pairs.append(dict(kind="clone_diff_speaker", cond="C1", item=a, ref_item=b, speaker=SPK[a],
                              sentence=s, repeat=r, test=p, ref=f"{W}/D1/{a}.wav"))
same, diffsp = [], []
for a, b in itertools.combinations(ITEMS, 2):
    (same if SPK[a] == SPK[b] else diffsp).append((a, b))
for a, b in same:
    pairs.append(dict(kind="real_same_speaker", item=a, ref_item=b, speaker=SPK[a],
                      test=f"{W}/D1/{a}.wav", ref=f"{W}/D1/{b}.wav"))
for a, b in diffsp:                                   # 전체 조합 사용 (25명이면 300쌍)
    pairs.append(dict(kind="real_diff_speaker", item=a, ref_item=b, speaker=SPK[a],
                      test=f"{W}/D1/{a}.wav", ref=f"{W}/D1/{b}.wav"))
KINDS = ("clone", "input", "clone_diff_speaker", "real_same_speaker", "real_diff_speaker")
print("비교 쌍:", {k: sum(1 for p in pairs if p["kind"] == k) for k in KINDS})
assert any(p["kind"] == "clone" for p in pairs), "복제본이 없습니다. 9단계를 확인하세요."
RESULT_CSV = f"{DRIVE_ROOT}/runs/{RUN_NAME}/sv_results.csv"
json.dump({"out_csv": RESULT_CSV, "pairs": pairs}, open("/content/sv_cfg.json", "w"), ensure_ascii=False)

In [ ]:
!conda run -n sv --no-capture-output python /content/sv_eval2.py /content/sv_cfg.json

## 12단계. 점수 눈금 만들기

**이번 데이터는 EER을 계산할 수 있다.** 화자당 클립이 13~18개라 같은 화자 쌍(genuine)이 약 475개 생긴다. 원본 가이드는 화자당 녹음 1개를 가정해 EER을 포기했지만 우리는 낼 수 있다.

- **EER 임계값**: 같은 화자 쌍과 다른 화자 쌍이 교차하는 지점. 세 방법 비교에 쓸 공통 임계값은 이 값으로 정한다.
- **FAR 5% 임계값**: 다른 화자 점수의 95번째 백분위수. EER과 함께 보면 해석이 단단해진다.
- `*_pass_ref`의 0.25 / 0.697은 **참고선일 뿐 보정된 값이 아니다.** 0.25는 SpeechBrain 기본값이고, Enkidu는 0.2922, VocalBridge는 0.419를 EER로 보고했다. 모델과 데이터마다 다르다.

여기서 나온 EER 임계값을 팀원들과 공유해 **세 보호 기법이 같은 값을 쓰게** 해야 비교가 성립한다.

In [ ]:
import pandas as pd, numpy as np
from IPython.display import display

df = pd.read_csv(RESULT_CSV)
if len(ITEMS) < 63:
    print(f"주의: 항목 {len(ITEMS)}개로 분석 중입니다(시험 모드). 본 결과는 N_ITEMS = None으로 다시 보세요.\n")

cal = df[df.kind.isin(["real_same_speaker", "real_diff_speaker", "clone_diff_speaker"])]
display(cal.groupby("kind")[["ecapa", "resemblyzer"]].agg(["count", "mean", "std", "min", "max"]).round(3))

def eer_threshold(same, diff):
    best = None
    for t in np.unique(np.concatenate([same, diff])):
        frr = np.mean(same < t); far = np.mean(diff >= t)
        if best is None or abs(frr - far) < best[0]:
            best = (abs(frr - far), float(t), float((frr + far) / 2))
    return best[1], best[2]

EER_THR = {}
s = df[df.kind == "real_same_speaker"]; d = df[df.kind == "real_diff_speaker"]
print()
if len(s) >= 5 and len(d) >= 5:
    for m in ("ecapa", "resemblyzer"):
        t, e = eer_threshold(s[m].values, d[m].values)
        EER_THR[m] = t
        print(f"{m}: EER 임계값 {t:.4f}  (EER {e:.2%}, 같은화자 {len(s)}쌍 / 다른화자 {len(d)}쌍)")
    print("\n  ^ 이 값을 팀원들과 공유해 세 기법이 같은 임계값을 쓰게 하세요.")
else:
    print("같은 화자 쌍이 부족해 EER을 계산하지 못했습니다. FAR 기준만 씁니다.")

far_rows, THR = [], {}
for m in ("ecapa", "resemblyzer"):
    for kind in ("clone_diff_speaker", "real_diff_speaker"):
        v = df[df.kind == kind][m].values
        for far in (0.05, 0.01):
            if len(v) >= 10 / far:
                t = float(np.quantile(v, 1 - far))
                far_rows.append(dict(모델=m, 기준분포=kind, n=len(v), 목표FAR=far, 임계값=round(t, 4)))
                if kind == "clone_diff_speaker" and far == 0.05:
                    THR[m] = t
    if m not in THR:
        v = df[df.kind == "real_diff_speaker"][m].values
        if len(v) >= 200:
            THR[m] = float(np.quantile(v, 0.95))

if far_rows:
    print()
    display(pd.DataFrame(far_rows))
print("FAR 5% 임계값(THR):", {k: round(v, 4) for k, v in THR.items()} or "없음")
print("EER 임계값(EER_THR):", {k: round(v, 4) for k, v in EER_THR.items()} or "없음")
print("참고선(보정 안 됨): ecapa 0.25 = SpeechBrain 기본, resemblyzer 0.697 = De-AntiFake 논문")

## 13단계. 조건별 요약 (C1 원본 / C2 보호 / C3 정화 / C4 정화만)

In [ ]:
cl = df[df.kind == "clone"].copy()
for m in ("ecapa", "resemblyzer"):
    if m in THR:
        cl[f"{m}_pass_far5"] = cl[m] >= THR[m]
    if m in EER_THR:
        cl[f"{m}_pass_eer"] = cl[m] >= EER_THR[m]

agg = dict(n=("ecapa", "size"),
           ecapa_mean=("ecapa", "mean"), ecapa_sd=("ecapa", "std"),
           ecapa_min=("ecapa", "min"), ecapa_max=("ecapa", "max"),
           res_mean=("resemblyzer", "mean"), res_sd=("resemblyzer", "std"),
           res_min=("resemblyzer", "min"), res_max=("resemblyzer", "max"))
for m, short in (("ecapa", "ecapa"), ("resemblyzer", "res")):
    if f"{m}_pass_eer" in cl:
        agg[f"{short}_pass_eer"] = (f"{m}_pass_eer", "mean")
    if f"{m}_pass_far5" in cl:
        agg[f"{short}_pass_far5"] = (f"{m}_pass_far5", "mean")
    agg[f"{short}_pass_ref"] = (f"{m}_pass_ref", "mean")

g = cl.groupby("cond").agg(**agg).reindex(CONDS_CL).round(3)
display(g)

- `*_pass_eer`: **EER 임계값**을 넘은 비율. 세 방법 비교의 주 지표로 쓴다. 논문의 SPR/DPR과 가장 가까운 형태다.
- `*_pass_far5`: 다른 화자 분포 FAR 5% 기준. EER과 같이 보면 해석이 단단해진다.
- `*_pass_ref`: 0.25 / 0.697 참고선. **보정된 값이 아니므로** 논문과 형식만 맞춰 보는 용도다.

통과율보다 **평균 점수와 14단계의 쌍체 차이**를 먼저 보라. 임계값 근처의 통과/탈락은 불안정하다.

## 14단계. 항목별 쌍체 분석

같은 항목끼리 비교하므로 항목별 차이를 본다. 문장·반복은 항목 안에서 먼저 평균한다.

| 비교 | 읽는 법 |
| --- | --- |
| C1 − C2 | 보호로 떨어진 점수 (양수일수록 보호 효과) |
| C3 − C2 | 정화로 회복된 점수 (양수일수록 정화가 보호를 되돌림) |
| C1 − C3 | 정화 후에도 남은 하락 (0에 가까우면 원본 수준 회복) |
| C4 − C3 | 정화 후 남은 보호 효과. 0에 가까우면 정화가 보호를 완전히 무력화 |
| C1 − C4 | 정화 모델 자체의 손상. 보호가 없어도 정화만으로 떨어지는 양 |

`C1 − C3 = (C1 − C4) + (C4 − C3)`로 나뉘어, 남은 하락이 **정화 모델 탓인지 보호가 살아남은 탓인지** 구분된다. 이게 이번 실험의 핵심 분해다.

### ⚠️ 화자가 4명뿐이다

항목은 63개지만 독립 단위는 **화자 4명**이다. 화자 단위 클러스터 부트스트랩은 4개를 재표집하므로 신뢰구간이 거의 의미가 없다.

그래서 두 가지를 같이 낸다.

- `CI95_화자`: 화자 단위 부트스트랩. **보수적이지만 4개 클러스터라 불안정**
- `CI95_항목`: 항목 단위 부트스트랩. 좁게 나오지만 같은 화자 항목끼리 독립이 아니라 **실제보다 좁다**

진짜 구간은 둘 사이 어딘가다. 포스터에는 항목 단위를 싣되 "화자 4명" 한계를 명시하라. 화자를 8명 이상으로 늘리면 이 문제가 사라진다.

In [ ]:
from scipy.stats import wilcoxon

per = cl.groupby(["item", "speaker", "cond"])[["ecapa", "resemblyzer"]].mean().reset_index()
spk_of_item = per.drop_duplicates("item").set_index("item")["speaker"]
if len(ITEMS) < 63:
    print(f"주의: 항목 {len(ITEMS)}개(시험 모드) 결과입니다.\n")

def paired(metric, a, b, nboot=10000, seed=0):
    wide = per.pivot(index="item", columns="cond", values=metric)
    d = (wide[a] - wide[b]).dropna()
    sp = spk_of_item.loc[d.index]
    rng = np.random.default_rng(seed)

    # (1) 화자 단위 - 보수적이나 클러스터 4개라 불안정
    groups = [d[sp == s].values for s in sp.unique()]
    boots_s = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean()
               for _ in range(nboot)]
    lo_s, hi_s = np.percentile(boots_s, [2.5, 97.5])

    # (2) 항목 단위 - 좁지만 항목끼리 독립이 아님
    boots_i = rng.choice(d.values, size=(nboot, len(d)), replace=True).mean(axis=1)
    lo_i, hi_i = np.percentile(boots_i, [2.5, 97.5])

    try:
        p = wilcoxon(d.values).pvalue if len(d) >= 6 and np.any(d.values != 0) else float("nan")
    except Exception:
        p = float("nan")
    sd = d.std(ddof=1)
    return dict(비교=f"{a} - {b}", 항목수=len(d), 화자수=len(groups),
                평균차이=round(float(d.mean()), 4),
                CI95_항목=f"[{lo_i:.4f}, {hi_i:.4f}]",
                CI95_화자=f"[{lo_s:.4f}, {hi_s:.4f}]",
                d_z=round(float(d.mean() / sd), 2) if sd > 0 else float("nan"),
                차이가_양수인_항목=int((d > 0).sum()), wilcoxon_p=p)

def holm(ps):
    ps = np.asarray(ps, float); out = np.full(len(ps), np.nan)
    ok = ~np.isnan(ps); idx = np.argsort(ps[ok]); m = ok.sum()
    adj = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * ps[ok][idx]))
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

COMPARISONS = [("C1", "C2"), ("C3", "C2"), ("C1", "C3")]
if "C4" in CONDS_CL:
    COMPARISONS += [("C4", "C3"), ("C1", "C4")]

PAIRED = {}
for m, title in (("ecapa", "ECAPA"), ("resemblyzer", "Resemblyzer")):
    t = pd.DataFrame([paired(m, a, b) for a, b in COMPARISONS])
    t["p_holm"] = holm(t["wilcoxon_p"]).round(4)
    t["wilcoxon_p"] = t["wilcoxon_p"].round(4)
    PAIRED[m] = t
    print(title); display(t)

## 15단계. 그림 (항목별 평균 점수)

In [ ]:
import matplotlib.pyplot as plt

LABEL = {"C1": "C1 original", "C2": "C2 protected", "C3": "C3 purified", "C4": "C4 purify-only"}
fig, axes = plt.subplots(1, 2, figsize=(5 + 2 * len(CONDS_CL), 4.8))

for ax, (m, ref_thr, title) in zip(axes, [("ecapa", 0.25, "ECAPA"), ("resemblyzer", 0.697, "Resemblyzer")]):
    data = [per[per.cond == c][m].values for c in CONDS_CL]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(CONDS_CL) + 1))
    ax.set_xticklabels([LABEL[c] for c in CONDS_CL], fontsize=8)

    rng = np.random.default_rng(0)
    for i, v in enumerate(data, 1):
        ax.scatter(rng.normal(i, 0.04, len(v)), v, s=14, alpha=0.5)

    wide = per.pivot(index="item", columns="cond", values=m)[CONDS_CL]
    for _, row in wide.iterrows():
        ax.plot(range(1, len(CONDS_CL) + 1), row.values, color="gray", alpha=0.12, lw=0.7)

    if m in EER_THR:
        ax.axhline(EER_THR[m], ls="--", color="black", lw=1.6,
                   label=f"EER thr (our data) {EER_THR[m]:.3f}")
    if m in THR:
        ax.axhline(THR[m], ls="--", color="tab:blue", lw=1.1,
                   label=f"FAR 5% thr {THR[m]:.3f}")
    ax.axhline(ref_thr, ls=":", color="gray", label=f"paper ref {ref_thr} (uncalibrated)")
    for kind, ls, col in (("clone_diff_speaker", "-.", "tab:red"),
                          ("real_same_speaker", ":", "tab:green")):
        sub = df[df.kind == kind][m]
        if len(sub):
            ax.axhline(sub.mean(), ls=ls, color=col, lw=0.9, label=f"mean {kind}")

    ax.set_title(title); ax.set_ylabel("cosine similarity")
    ax.legend(fontsize=6.5, loc="lower left")

plt.tight_layout()
FIG = f"{DRIVE_ROOT}/runs/{RUN_NAME}/scores.png"
plt.savefig(FIG, dpi=200, bbox_inches="tight")
print("저장:", FIG)
plt.show()

In [ ]:
low = per[(per.cond == "C4") & (per.ecapa < 0.25)]
print(low[["item", "speaker", "ecapa"]].to_string())

## 16단계. (참고) 입력 음성 자체의 변화 (복제 전)
보호(D2), 정화(D3), 정화만(D4)이 입력의 화자 특성을 신호 수준에서 얼마나 바꿨는지 봅니다. `D1`은 자기 자신과의 비교라 1.0 근처가 나오는 확인용 행입니다. D4가 D1보다 많이 낮으면 정화 모델이 깨끗한 음성의 화자 특성도 깎는다는 뜻입니다.

In [ ]:
inp = df[df.kind == "input"]
display(inp.groupby("cond")[["ecapa", "resemblyzer"]].agg(["mean", "min", "max"]).reindex(CONDS_IN).round(3))

In [ ]:
import soundfile as sf
c4 = per[per.cond == "C4"].copy()
c4["dur"] = [sf.info(f"{W}/D1/{i}.wav").duration for i in c4["item"]]
print(c4[["item", "dur", "ecapa"]].sort_values("dur").head(10).to_string())
print("\n길이-점수 상관:", round(c4["dur"].corr(c4["ecapa"]), 3))

## 결과 해석 틀

| 관측 | 의미 |
|---|---|
| C1 대비 C2가 눈에 띄게 낮음(신뢰구간이 0을 포함하지 않음) | 보호가 이 모델에서 정화 전에 효과가 있음 |
| C2도 C1과 비슷함 | 이 보호는 XTTS-v2 복제를 막지 못함 |
| C3가 C2보다 높고 C1에 가까움 | 정화로 공격이 회복됨 |
| C3가 C2와 비슷하게 낮음 | 정화 후에도 보호가 남았거나, 정화가 복제 자체를 망가뜨린 것 → C4로 구분 |
| (D4) C4 ≈ C1, C3 < C4 | 정화 모델은 멀쩡한데 **보호가 정화를 버팀** (보호 기법에 유리한 결과) |
| (D4) C4 ≈ C3 < C1 | 남은 하락은 **정화 모델 자체의 손상**. 보호는 정화로 무력화됨 |
| (D4) C3 ≈ C4 ≈ C1 | 정화가 보호를 완전히 무력화 |
| ECAPA와 Resemblyzer가 엇갈림 | 한쪽만 믿지 말고 결론 보류 |

**꼭 알아둘 한계**
1. **기준이 D1 원본 그대로입니다.** C1은 D1 파일을 참조로 만든 복제인데 기준도 같은 D1 파일이라, C1은 "같은 녹음을 보고 만든 복제"의 상한에 가깝습니다. 연구 질문(보호된 음성으로 원래 화자를 얼마나 재현하는가)에는 맞는 구조이고 C2~C4도 같은 D1 녹음에서 나왔으므로 **조건 간 차이 비교는 유효**하지만, 절대 점수를 "독립 녹음 기준 화자 인증 성공률"로 부르지는 마세요. 같은 화자의 다른 녹음이 생기면 그것을 기준으로 다시 재는 것이 가장 공정합니다.
2. **D4(정화만)가 없으면** C3가 낮을 때 "보호가 남아서"인지 "정화가 음성을 망가뜨려서"인지 구분할 수 없습니다. `USE_D4 = True`를 권장합니다. D4는 D3와 같은 정화 설정으로 만들어야 합니다.
3. **ECAPA는 보호 기법이 최적화에 쓴 모델과 같은 계열일 수 있습니다**(기법마다 다름). 해당 기법의 논문에서 어떤 화자 모델로 최적화했는지 확인하세요. 독립 지표는 Resemblyzer입니다.
4. 0.25 / 0.697은 참고선입니다. 화자당 녹음이 1개라 EER은 계산할 수 없고, 우리 데이터 임계값은 다른 화자 분포의 FAR 5% 기준입니다(FRR은 알 수 없음).
5. 25개는 효과 크기가 작을 때 부족할 수 있습니다. 신뢰구간이 넓으면 항목을 늘려야 합니다.
6. 이 비교는 보호 기법을 모르는 **비적응형** 공격 기준입니다. 정화 설정(예: De-AntiFake의 `t`)을 바꾼 적응형 공격은 별도 실험입니다.

## 자주 막히는 곳

| 증상 | 가능한 원인과 해결 |
|---|---|
| 4단계에서 `AssertionError` | 약관을 읽고 동의할 때만 `AGREE_CPML = True`로 바꾸세요 |
| 복제 중 `EOFError` 또는 약관 입력 요구 | `COQUI_TOS_AGREED`가 전달되지 않음. 4단계를 다시 실행하고 같은 세션에서 복제 칸 실행 |
| `Weights only load failed` / `UnpicklingError` | torch 신버전 문제. 2단계의 `torch==2.5.1` 고정이 적용됐는지 3단계로 확인 (미확인) |
| `'GPT2InferenceModel' object has no attribute 'generate'` | `transformers`가 너무 최신. `transformers<4.50`이 적용됐는지 확인 (미확인) |
| `No module named 'pkg_resources'` | `pip install "setuptools<81"` (2단계에 포함) |
| `D1에 없는 항목` 목록이 나옴 | D2, D3 파일 이름이 D1과 다름. 이름 꼬리표는 `STRIP`으로 제거 |
| 복제 출력이 없고 오류도 안 보임 | `conda run`이 출력을 숨김. 9단계의 개수와 `failed.json`을 확인 |
| 길이 경고 | D1~D4가 같은 클립에서 만든 것이 맞는지 확인 |
| `중단: ... 설정이 다릅니다` / `참조 음성이 이전 실행과 다른 항목` | 같은 RUN_NAME에 다른 입력이나 문장으로 만든 복제본이 있음. RUN_NAME을 바꾸세요 |
| 손실 압축 경고 | D2~D4를 wav/flac으로 다시 저장 |
| 9단계의 샘플레이트가 24000이 아님 | 모델이 다른 것으로 로드됨. 3단계와 `clone_batch.py`의 모델 이름 확인 |
| `[검증]` 줄의 두 점수가 크게 다름 | 임베딩 직접 계산에 문제가 있음. 12~14단계 해석 전에 알려 주세요 |
| `/content`의 파일, conda 환경이 사라짐 | 세션 초기화. 1단계부터 다시. 복제본과 결과 CSV는 드라이브(`DRIVE_ROOT/runs/RUN_NAME`)에 남아 있고, 복제 칸은 이어서 진행됨 |

## 17단계. 결과 저장
복제본과 `sv_results.csv`는 이미 드라이브(`DRIVE_ROOT/runs/RUN_NAME`)에 저장돼 있습니다. 아래는 노트북 내려받기용 zip이 필요할 때만 쓰세요.

In [ ]:
!cd {DRIVE_ROOT}/runs && zip -rq /content/{RUN_NAME}_results.zip {RUN_NAME}
from google.colab import files
files.download(f"/content/{RUN_NAME}_results.zip")